# 动态模型（dynamic_model）
动态模型在 运行时 根据当前 状态 和上下文进行选择。这支持复杂的路由逻辑和成本优化。
要使用动态模型，请使用 @wrap_model_call 装饰器创建中间件，以修改请求中的模型

你写一个函数（动态选模型、改参数、加日志）
头上加 @wrap_model_call
→ 等于告诉 LangChain：“每次调用模型之前，都先跑我这个函数！”
LangChain 自动把它变成中间件，每次请求都会走这里

@wrap_model_call = 注册中间件的装饰器
加了它 = 每次模型调用都会先经过你这个函数
handler = call_next () 继续执行

In [4]:
import os
from langchain_openai import ChatOpenAI
from langchain.chat_models import init_chat_model

from dotenv import load_dotenv

load_dotenv()

basic_model = init_chat_model(
    model="qwen-plus",  # 模型名称可自定义
    model_provider="openai",  # 如果是Langchain不支持的模型类型，需要指定模型提供者，虽然Langchain不支持阿里千问，但是千问兼容openai
    base_url=os.getenv("DASHSCOPE_BASE_URL"),
    api_key=os.getenv("DASHSCOPE_API_KEY"),
)
advanced_model = init_chat_model(
    model="qwen3.5-plus",  # 模型名称可自定义
    model_provider="openai",  # 如果是Langchain不支持的模型类型，需要指定模型提供者，虽然Langchain不支持阿里千问，但是千问兼容openai
    base_url=os.getenv("DASHSCOPE_BASE_URL"),
    api_key=os.getenv("DASHSCOPE_API_KEY"),
)

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import ModelRequest, ModelResponse, wrap_model_call

@wrap_model_call
def dynamic_model_selection(request: ModelRequest, handler) -> ModelResponse:
    """根据对话复杂性选择模型。"""
    message_count = len(request.state["messages"])

    if message_count > 10:
        # 对较长的对话使用高级模型
        model = advanced_model
    else:
        model = basic_model

    # 把选好的模型塞回请求
    request.model = model
    # ✅ 关键：交给 handler 真正执行
    return handler(request)

agent = create_agent(
    model=basic_model,  # 默认模型
    # tools=tools,
    middleware=[dynamic_model_selection]
)

1. handler 从哪来？谁传的？
    LangGraph 自动传给你的！你不用定义、不用传、不用管来源。
2. handler 到底是什么类型？
    它是一个 可调用函数（callable）
    handler(request: ModelRequest) -> ModelResponse
3. 我必须调用它吗？
    必须！不调用 → 模型不跑 → 流程卡住。

In [ ]:
# 中间件模式（FastAPI / Starlette）
async def middleware(request, call_next):
    # 你自己的逻辑
    response = await call_next(request)  # 交给下一个
    return response

# LangGraph 模型包装器
@wrap_model_call
def dynamic_model_selection(request, handler):
    # 你自己的逻辑
    response = handler(request)  # 交给下一个（真正调用模型）
    return response